<a href="https://colab.research.google.com/github/mkl-pkl/AERO2460---Truss-Design-Project/blob/main/AERO2460_loads.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# AERO2460/8460/9460 - 2026 Group Design Project – Truss Frame
# Notebook for generating load cases and computing loads at engine mount nodes.
# Author: Michael Keller
# Date: 18/10/2026

In [ ]:
# Import relevant packages.
import importlib.util
import subprocess
import numpy as np
import pandas as pd
import itertools

# Install ipympl (if not already installed)
if importlib.util.find_spec("ipympl") is None:
  subprocess.run(["pip", "install", "-q", "ipympl"])

import os
from google.colab import drive

# Mount drive (if not already mounted)
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")

import sys

# Add folder with helper scripts to our system path (avoiding duplicates)
hs_path = '/content/drive/MyDrive/AERO2460/GroupProject/HelperScripts'
if hs_path not in sys.path:
  sys.path.append(hs_path)

# Now we can import specific helper scripts
import load_cloud_cells as lcc
import free_body_plot as fb

importlib.reload(lcc)
importlib.reload(fb)

<module 'free_body_plot' from '/content/drive/MyDrive/AERO2460/GroupProject/HelperScripts/free_body_plot.py'>

In [ ]:
aircraft = {
    'w_gross': 1370,      # [lb]
    'n_max_pos':  3.8,    # [-]
    'n_max_neg':  -1.68,  # [-]
    'n_side_max': 1.33,   # [-] Side loads can be in +-y direction --> 2x load cases --> which is most conservative?
    'V_c_gust_max': 4.844,  # Cruise positive gust load factor
    'V_c_gust_min': -2.844, # Cruise negative gust load factor
    'V_d_gust_max': 3.691,  # Dive positive gust load factor
    'V_d_gust_min': -1.691, # Dive negative gust load factor
    'n_gyro': 2.5     # Load factor used for propeller gyroscopic load cases
}

engine = {
    'weight':     68,                           # [lb]
    'cg':         np.array([-7/12, 0, 0]),      # [ft]
    'thrust':     np.array([172, 0, 0]),        # [lb]  we assume thrust acts at the origin
    't_prop':     np.array([-206, 0, 0]),       # [lbft]
    't_gyro_y':   np.array([0, 25, 0]),         # [lbft] can be +/-
    't_gyro_z':   np.array([0, 0, 25]),         # [lbft] can be +/-
    'r_nodes':    np.array([
                          [0, -5/12, -4/12],    # Node A, [ft]
                          [0, -5/12, 4/12],     # Node B, [ft]
                          [0, 5/12, 4/12],      # Node C, [ft]
                          [0, 5/12, -4/12]      # Node D, [ft]
                          ])
}

In [ ]:
class LoadCase:

  def __init__(self, loads, moments):
    # Initialise dict to store load case details/identifiers
    self.details = {}

    # Combine applied loads and moments into single net applied load and moment vectors
    self.load_vec         = np.sum(loads, axis=0)
    self.moment_vec       = np.sum(moments, axis=0)

  def compute_nodal_loads(self, r_nodes):
    # Convert net applied force and moment into equivalent applied load at nodes A, B, C, D.

    # Create arrays to store nodal load contributions due to applied forces and moments
    self.p_nodes_f = np.zeros((4, 3))
    self.p_nodes_m = np.zeros((4, 3))
    self.p_nodes = np.zeros((4, 3))

    # Assume net applied FORCE is shared equally between all 4 nodes.
    self.p_nodes_f[0, :] = self.load_vec*0.25
    self.p_nodes_f[1, :] = self.load_vec*0.25
    self.p_nodes_f[2, :] = self.load_vec*0.25
    self.p_nodes_f[3, :] = self.load_vec*0.25

    # -------------------------------------------------------------------------
    # Assume net applied MOMENT is shared equally between all 4 nodes.
    # Assume all 4 nodes are pin-jointed (i.e. cannot resist moments)

    # Get distances from origin of each nodes (A, B, C, D) in each plane (YZ, ZX, XY)
    r_yz = np.linalg.norm(r_nodes[:, [1, 2]], axis=1)
    r_zx = np.linalg.norm(r_nodes[:, [2, 0]], axis=1)
    r_xy = np.linalg.norm(r_nodes[:, [0, 1]], axis=1)

    # Get x, y, z components of node positions.
    r_nodes_x = r_nodes[:, 0]
    r_nodes_y = r_nodes[:, 1]
    r_nodes_z = r_nodes[:, 2]

    # Compute angles to node positions.
    theta_yz = np.arctan2(r_nodes_z, r_nodes_y)
    theta_zx = np.arctan2(r_nodes_x, r_nodes_z)
    theta_xy = np.arctan2(r_nodes_y, r_nodes_x)

    # Compute unit tangent vector for each angle U_t = [-sin(theta), cos(theta)]
    ut_yz = np.array([-np.sin(theta_yz), np.cos(theta_yz)])
    ut_xz = np.array([-np.sin(theta_zx), np.cos(theta_zx)])
    ut_xy = np.array([-np.sin(theta_xy), np.cos(theta_xy)])

    # Compute force at each node due to moment
    p_nodes_yz = ut_yz * r_yz * self.moment_vec[0] / np.sum(r_yz**2, axis=0)
    p_nodes_zx = ut_xz * r_zx * self.moment_vec[1] / np.sum(r_zx**2, axis=0)
    p_nodes_xy = ut_xy * r_xy * self.moment_vec[2] / np.sum(r_xy**2, axis=0)

    # Collect loads in each direction
    self.p_nodes_m[:, 0] = p_nodes_zx[1, :] + p_nodes_xy[0, :]
    self.p_nodes_m[:, 1] = p_nodes_yz[0, :] + p_nodes_xy[1, :]
    self.p_nodes_m[:, 2] = p_nodes_yz[1, :] + p_nodes_zx[0, :]

    # ------------------------------------------------------------------
    # Now sum the contributions from both direct loads and moments
    self.p_nodes = self.p_nodes_f + self.p_nodes_m

# ------------------------------------------------------------------
# TEST load case --> VERIFYING RESULTS
test_lc = LoadCase(
    loads = np.array([[0, 0, 0], [0, 0, 0]]),
    moments = np.array([engine['t_prop']])
)

test_lc.compute_nodal_loads(r_nodes = engine['r_nodes'])

In [ ]:
# # So what are the different load cases we want to investigate?
# # We need to know what the most conservative load cases are.

def make_case(name, n, thrust_on=1, torque_on=1, gy_on=0, gz_on=0):
    """
    name          : load case name/identifier
    n             : load-factor vector [nx, ny, nz]
    thrust_on     : 0/1 switch on engine['thrust']
    torque_on     : multiplier on engine['t_prop'] (0 = off, 1 = as given, k = 6.1.3 factor)
    gy_on, gz_on  : -1/0/+1 sign multipliers on the gyroscopic moments about Y and Z
    """

    # Compute the inertial loads due to load factor n
    f_inertial = engine['weight'] * n

    # Compute moments about origin O, generated by inertial loads
    m_inertial = np.cross(engine['cg'], f_inertial)

    # Create the LoadCase instance
    lc = LoadCase(
        loads   = np.array([f_inertial, thrust_on * engine['thrust']]),
        moments = np.array([
            m_inertial,
            torque_on * engine['t_prop'],
            gy_on * engine['t_gyro_y'],
            gz_on * engine['t_gyro_z']])
    )
    # Store the load case input details.
    lc.details = {
        'Name': name,
        'n_z': n[2],
        'n_y': n[1],
        'Thrust [lb]': thrust_on * engine['thrust'][0],
        'Torque X [lbft]': torque_on * engine['t_prop'][0],
        'Gyro Y [lbft]': gy_on * engine['t_gyro_y'][1],
        'Gyro Z [lbft]': gz_on * engine['t_gyro_z'][2]
        }
    return lc

In [ ]:
# Now define the full list of load cases.
cases = [
    make_case(name = 'Gust +Vc', n = np.array([0, 0, aircraft['V_c_gust_max']])),
    make_case(name = 'Gust -Vc', n = np.array([0, 0, aircraft['V_c_gust_min']])),
    make_case(name = 'Gust +Vd', n = np.array([0, 0, aircraft['V_d_gust_max']])),
    make_case(name = 'Gust -Vd', n = np.array([0, 0, aircraft['V_d_gust_min']])),
    make_case(name = 'Side +Y', n = np.array([0, aircraft['n_side_max'], 1])),
    make_case(name = 'Side -Y', n = np.array([0, -aircraft['n_side_max'], 1]))
]

# Define all possible propeller gyroscopic torque load cases.
cases += [make_case(
                    name  = f'Gyro Y{gy:+d} Z{gz:+d}',
                    n     = np.array([0, 0, aircraft['n_gyro']]),
                    gy_on    = gy,
                    gz_on    = gz
                    ) for gy, gz in itertools.product((-1, 0, 1), repeat=2) if (gy, gz) != (0, 0)]

# ------------------------------------------------------------------------------------------
# Now compute loads generated at each of the four engine mount nodes A, B, C, D
for lc in cases: lc.compute_nodal_loads(engine['r_nodes'])

# Generate a table with a summary of the load cases
case_table = pd.DataFrame([lc.details for lc in cases])
node_mag   = pd.DataFrame({lc.details['Name']: np.linalg.norm(lc.p_nodes, axis=1) for lc in cases},
                          index=list('ABCD')).T # |P| per node [lb]

node_names = ['A', 'B', 'C', 'D']
nodal_df = pd.DataFrame(
    [c.p_nodes.ravel() for c in cases], # row: Ax Ay Az Bx By Bz Cx Cy Cz Dx Dy Dz [lb]
    index   = pd.Index([c.details['Name'] for c in cases], name='Load case'),
    columns = [f'{n}{ax}' for n in node_names for ax in 'xyz'])

# case_table
# node_mag
nodal_df

,Ax,Ay,Az,Bx,By,Bz,Cx,Cy,Cz,Dx,Dy,Dz
Load case,,,,,,,,,,,,
Gust +Vc,-101.10900,-60.292683,157.713854,187.10900,60.292683,157.713854,187.10900,60.292683,6.982146,-101.10900,-60.292683,6.982146
Gust -Vc,127.60900,-60.292683,27.017854,-41.60900,60.292683,27.017854,-41.60900,60.292683,-123.713854,127.60900,-60.292683,-123.713854
Gust +Vd,-66.80725,-60.292683,138.112854,152.80725,60.292683,138.112854,152.80725,60.292683,-12.618854,-66.80725,-60.292683,-12.618854
Gust -Vd,93.30725,-60.292683,46.618854,-7.30725,60.292683,46.618854,-7.30725,60.292683,-104.112854,93.30725,-60.292683,-104.112854
Side +Y,-18.40400,-37.682683,92.365854,41.09600,82.902683,92.365854,104.40400,82.902683,-58.365854,44.90400,-37.682683,-58.365854
Side -Y,44.90400,-82.902683,92.365854,104.40400,37.682683,92.365854,41.09600,37.682683,-58.365854,-18.40400,-82.902683,-58.365854
Gyro Y-1 Z-1,-27.62500,-60.292683,117.865854,83.62500,60.292683,117.865854,113.62500,60.292683,-32.865854,2.37500,-60.292683,-32.865854
Gyro Y-1 Z+0,-12.62500,-60.292683,117.865854,98.62500,60.292683,117.865854,98.62500,60.292683,-32.865854,-12.62500,-60.292683,-32.865854
Gyro Y-1 Z+1,2.37500,-60.292683,117.865854,113.62500,60.292683,117.865854,83.62500,60.292683,-32.865854,-27.62500,-60.292683,-32.865854


In [ ]:
# -----------------------------------------------------------------------------
# We then want to generate some kind of plot to visualise all of the load cases to compare them against each other
# Maybe some kind of heatmap, showing each load case on x axis, nodes on y axis, plot values then being the nodal load magnitude.
# We can then highlight cases where each nodal load is highest, and those where the overall loads are highest


# lcc.plot_load_cloud_3d(cases, rays='hull', show_edges=False).show()
# lcc.plot_load_cloud_2d(cases).show()


In [ ]:
# Now looking at the loads in 3D
fb.plot_free_body_3d(cases, engine['r_nodes'], show_arrows=True, show_points=False, show_edges=False).show()